# 02 - Configure and Publish Function

This notebook publishes the Azure Function code into the app created by Notebook 1, writes the function key into `env/.env`, and then prepares Speech SDK options for the roundtrip test.

**Pre-requisite:** run `01_deploy_infra.ipynb` first so `env/.env` exists and contains the Function App resource details.

---

## Step 1 - Load environment variables

In [17]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

required = [
    "AZURE_RESOURCE_GROUP",
    "AZURE_AI_ENDPOINT",
    "AZURE_AUTH_MODE",
    "AZURE_FUNCTION_APP_NAME",
    "AZURE_FUNCTION_URL",
    "AZURE_APP_INSIGHTS_NAME",
]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Missing environment variables: {missing}")

if os.environ["AZURE_AUTH_MODE"].lower() != "aad":
    raise RuntimeError("This demo supports managed identity / AAD mode only.")

print("Environment loaded.")
print(f"Resource group: {os.environ['AZURE_RESOURCE_GROUP']}")
print(f"AI endpoint   : {os.environ['AZURE_AI_ENDPOINT']}")
print(f"Function app  : {os.environ['AZURE_FUNCTION_APP_NAME']}")
print(f"Function URL  : {os.environ['AZURE_FUNCTION_URL']}")
print(f"Auth mode     : {os.environ['AZURE_AUTH_MODE']}")

Environment loaded.
Resource group: rg-speech01-foundry
AI endpoint   : https://aispeech016m4wo.cognitiveservices.azure.com/
Function app  : func-speech01-6m4wo
Function URL  : https://func-speech01-6m4wo.azurewebsites.net/api/speech-roundtrip
Auth mode     : aad


## Step 2 - Publish function code and fetch the key

Run these two sub-steps in order:

1. **Step 2A:** Build the deployment ZIP package and prepare ARM auth.
2. **Step 2B:** Upload the package to blob storage, set `WEBSITE_RUN_FROM_PACKAGE` to the blob URL, and write the function app key into `env/.env`.

This follows Linux Consumption deploy-from-URL guidance for zip package deployment.

In [24]:
# Step 2A - Build function package (with Linux Python deps) and prepare ARM auth
import os
import subprocess
import tempfile
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import requests
from azure.identity import AzureCliCredential, DefaultAzureCredential

resource_group = os.environ["AZURE_RESOURCE_GROUP"]
function_app_name = os.environ["AZURE_FUNCTION_APP_NAME"]
function_app_source = Path("../functionapp")
if not function_app_source.exists():
    raise RuntimeError(f"Function app source folder not found: {function_app_source}")

try:
    credential = AzureCliCredential()
    management_token = credential.get_token("https://management.azure.com/.default").token
    credential_source = "AzureCliCredential"
except Exception:
    credential = DefaultAzureCredential()
    management_token = credential.get_token("https://management.azure.com/.default").token
    credential_source = "DefaultAzureCredential"

management_headers = {
    "Authorization": f"Bearer {management_token}",
    "Content-Type": "application/json",
}

preferred_subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID", "").strip()
subscriptions_resp = requests.get(
    "https://management.azure.com/subscriptions",
    headers=management_headers,
    params={"api-version": "2020-01-01"},
    timeout=60,
)
if subscriptions_resp.status_code < 200 or subscriptions_resp.status_code >= 300:
    raise RuntimeError(
        "Failed to discover subscriptions via ARM.\n"
        f"HTTP: {subscriptions_resp.status_code}\n"
        f"Response body: {subscriptions_resp.text}"
    )

subscriptions = subscriptions_resp.json().get("value", [])
enabled_subscriptions = [
    s.get("subscriptionId")
    for s in subscriptions
    if str(s.get("state", "")).lower() == "enabled" and s.get("subscriptionId")
]
if not enabled_subscriptions:
    raise RuntimeError("No enabled Azure subscription found for current credential.")

candidate_subscriptions = []
if preferred_subscription_id:
    candidate_subscriptions.append(preferred_subscription_id)
for sub_id in enabled_subscriptions:
    if sub_id not in candidate_subscriptions:
        candidate_subscriptions.append(sub_id)

subscription_id = None
for sub_id in candidate_subscriptions:
    rg_resp = requests.get(
        f"https://management.azure.com/subscriptions/{sub_id}/resourcegroups/{resource_group}",
        headers=management_headers,
        params={"api-version": "2022-09-01"},
        timeout=60,
    )
    if rg_resp.status_code == 200:
        subscription_id = sub_id
        break

if not subscription_id:
    raise RuntimeError(
        f"Resource group '{resource_group}' was not found in any enabled subscription visible to this credential.\n"
        "Set AZURE_SUBSCRIPTION_ID in ../env/.env to the correct subscription and rerun this step."
    )

base_site_url = (
    f"https://management.azure.com/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}/providers/Microsoft.Web/sites/{function_app_name}"
)


def arm_request(method: str, relative_path: str, payload: dict | None = None):
    response = requests.request(
        method=method,
        url=f"{base_site_url}{relative_path}",
        headers=management_headers,
        params={"api-version": "2022-03-01"},
        json=payload,
        timeout=60,
    )
    if response.status_code < 200 or response.status_code >= 300:
        raise RuntimeError(
            f"ARM {method} {relative_path} failed with HTTP {response.status_code}.\n"
            f"Response body: {response.text}"
        )
    if response.text.strip():
        return response.json()
    return {}


package_path = Path("../outputs/functionapp-package.zip")
package_path.parent.mkdir(parents=True, exist_ok=True)
if package_path.exists():
    package_path.unlink()

with tempfile.TemporaryDirectory() as temp_dir:
    staging_dir = Path(temp_dir) / "functionapp"
    staging_dir.mkdir(parents=True, exist_ok=True)

    for source_path in function_app_source.rglob("*"):
        if source_path.is_file():
            destination = staging_dir / source_path.relative_to(function_app_source)
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(source_path.read_bytes())

    requirements_path = staging_dir / "requirements.txt"
    python_packages_dir = staging_dir / ".python_packages" / "lib" / "site-packages"
    python_packages_dir.mkdir(parents=True, exist_ok=True)

    install_result = subprocess.run(
        [
            Path(os.sys.executable),
            "-m",
            "pip",
            "install",
            "--requirement",
            str(requirements_path),
            "--target",
            str(python_packages_dir),
            "--upgrade",
            "--only-binary=:all:",
            "--platform",
            "manylinux2014_x86_64",
            "--implementation",
            "cp",
            "--python-version",
            "3.12",
            "--abi",
            "cp312",
        ],
        capture_output=True,
        text=True,
    )
    print(install_result.stdout)
    if install_result.returncode != 0:
        raise RuntimeError(
            "Failed to build Linux-compatible dependencies for package deployment.\n"
            f"stderr: {install_result.stderr.strip()}\n"
            f"stdout: {install_result.stdout.strip()}"
        )

    with zipfile.ZipFile(package_path, "w", zipfile.ZIP_DEFLATED) as zip_file:
        for file_path in staging_dir.rglob("*"):
            if file_path.is_file():
                zip_file.write(file_path, file_path.relative_to(staging_dir))

print(f"Created function package: {package_path.resolve()}")
print(f"Using credential source: {credential_source}")
print(f"Subscription used: {subscription_id}")
print("Step 2A complete. Run the next cell to upload package URL and fetch the function key.")

  Using cached azure_functions-1.24.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached azure_identity-1.25.3-py3-none-any.whl.metadata (91 kB)
  Using cached werkzeug-3.1.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
  Using cached azure_core-1.41.0-py3-none-any.whl.metadata (49 kB)
  Using cached cryptography-48.0.0-cp311-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.3 kB)
  Using cached msal-1.37.0-py3-none-any.whl.metadata (11 kB)
  Using cached msal_extensions-1.3.1-py3-none-any.whl.metadata (7.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached cffi-2.0.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached pyjwt-2.13.0-py3-none-any.

# Step 2B - Upload package, configure deploy-from-URL, and write key to env/.env
import time
from pathlib import Path

from azure.core.exceptions import HttpResponseError, ResourceExistsError
from azure.storage.blob import BlobServiceClient
from dotenv import set_key

app_settings_data = arm_request("POST", "/config/appsettings/list")
app_settings = app_settings_data.get("properties", {})
blob_service_uri = app_settings.get("AzureWebJobsStorage__blobServiceUri", "").strip()
if not blob_service_uri:
    raise RuntimeError(
        "Cannot determine storage account for deploy-from-URL. "
        "Expected AzureWebJobsStorage__blobServiceUri in Function App settings."
    )

storage_account_name = urlparse(blob_service_uri).netloc.split(".")[0]

deploy_container_name = "function-packages"
deploy_blob_name = f"{function_app_name}-{int(time.time())}.zip"

blob_service = BlobServiceClient(account_url=blob_service_uri, credential=credential)
container_client = blob_service.get_container_client(deploy_container_name)
try:
    try:
        container_client.create_container()
    except ResourceExistsError:
        pass

    with package_path.open("rb") as package_stream:
        container_client.upload_blob(name=deploy_blob_name, data=package_stream, overwrite=True)
except HttpResponseError as ex:
    error_code = str(getattr(ex, "error_code", "") or "")
    error_text = str(ex)
    raise RuntimeError(
        "Storage upload failed during package publish.\n"
        f"error_code: {error_code or 'Unknown'}\n"
        f"error: {error_text}"
    ) from ex

package_url = f"{blob_service_uri.rstrip('/')}/{deploy_container_name}/{deploy_blob_name}"
app_settings.update(
    {
        "WEBSITE_RUN_FROM_PACKAGE": package_url,
        "WEBSITE_RUN_FROM_PACKAGE_BLOB_MI_RESOURCE_ID": "SystemAssigned",
        "SCM_DO_BUILD_DURING_DEPLOYMENT": "false",
        "ENABLE_ORYX_BUILD": "false",
    }
)
arm_request("PUT", "/config/appsettings", {"properties": app_settings})
arm_request("POST", "/restart")
time.sleep(20)
arm_request("POST", "/syncfunctiontriggers")

# After restart, function registration can take a short propagation window.
deployed_functions = []
for _ in range(6):
    functions_data = arm_request("GET", "/functions")
    deployed_functions = functions_data.get("value", [])
    if deployed_functions:
        break
    time.sleep(10)

if not deployed_functions:
    raise RuntimeError(
        "Deployment package URL was set, but no functions were registered after restart and trigger sync.\n"
        f"package_url: {package_url}\n"
        "Check Function App log stream for import errors from package dependencies."
    )

host_keys = arm_request("POST", "/host/default/listKeys")
function_app_key = host_keys.get("functionKeys", {}).get("default") or host_keys.get("masterKey")
if not function_app_key:
    raise RuntimeError("ARM returned an empty function app host key.")

env_path = Path("../env/.env")
if not env_path.exists():
    raise RuntimeError("Expected ../env/.env to exist. Rerun Notebook 1 first.")

# Replace or append AZURE_FUNCTION_KEY in a single call.
set_key(str(env_path), "AZURE_FUNCTION_KEY", function_app_key, quote_mode="never")

function_names = [fn.get("name", "") for fn in deployed_functions if fn.get("name")]
print(f"Deploy-from-URL configured with package: {package_url}")
print(f"Storage account: {storage_account_name}")
print(f"Registered functions: {function_names}")
print(f"Function key written to: {env_path.resolve()}")
print(f"Function deploy and key setup complete for: {function_app_name}")

## Step 3 - Initialize Speech SDK config

Use Azure AD token authentication from `DefaultAzureCredential`.

In [26]:
import azure.cognitiveservices.speech as speechsdk
from azure.identity import DefaultAzureCredential

speech_endpoint = os.environ["AZURE_AI_ENDPOINT"]

credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default").token

speech_config = speechsdk.SpeechConfig(endpoint=speech_endpoint)
speech_config.authorization_token = token
speech_config.speech_recognition_language = "en-US"

print("Speech SDK initialized with AAD token.")

Speech SDK initialized with AAD token.


## Step 4 - Preview available voices

In [27]:
synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=None)
voices_result = synthesizer.get_voices_async("en-US").get()

if voices_result.reason != speechsdk.ResultReason.VoicesListRetrieved:
    raise RuntimeError(f"Failed to fetch voices. Reason: {voices_result.reason}")

voice_names = [voice.short_name for voice in voices_result.voices][:10]
print("Sample en-US voices:")
for name in voice_names:
    print(f"- {name}")

Sample en-US voices:
- en-US-Ava:DragonHDLatestNeural
- en-US-Andrew:DragonHDLatestNeural
- en-US-Adam:DragonHDLatestNeural
- en-US-Alloy:DragonHDLatestNeural
- en-US-Aria:DragonHDLatestNeural
- en-US-Bree:DragonHDLatestNeural
- en-US-Brian:DragonHDLatestNeural
- en-US-Davis:DragonHDLatestNeural
- en-US-Emma:DragonHDLatestNeural
- en-US-Emma2:DragonHDLatestNeural


In [28]:
default_voice = os.getenv("AZURE_SPEECH_VOICE", "en-US-AvaMultilingualNeural")
speech_config.speech_synthesis_voice_name = default_voice
print(f"Selected voice for test notebook: {default_voice}")

Selected voice for test notebook: en-US-AvaMultilingualNeural


## Step 6 - Confirm readiness for roundtrip test

In [29]:
readiness = {
    "ai_endpoint": os.environ["AZURE_AI_ENDPOINT"],
    "voice": speech_config.speech_synthesis_voice_name,
}
readiness

{'ai_endpoint': 'https://aispeech016m4wo.cognitiveservices.azure.com/',
 'voice': 'en-US-AvaMultilingualNeural'}

---

## ✅ Configuration complete

Proceed to **`03_test.ipynb`** for the speech synthesis and recognition roundtrip, and to **`04_function.ipynb`** for the function invocation flow.